In [ ]:
!pip install google-cloud-vision

In [ ]:
from google.cloud import vision

In [ ]:
def detect_text(path):
    """Detects text in the file."""
    from google.cloud import vision

    client = vision.ImageAnnotatorClient()

    with open(path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)

    response = client.text_detection(image=image)
    texts = response.text_annotations
    print("Texts:")

    for text in texts:
        print(f'\n"{text.description}"')

        vertices = [
            f"({vertex.x},{vertex.y})" for vertex in text.bounding_poly.vertices
        ]

        print("bounds: {}".format(",".join(vertices)))

    if response.error.message:
        raise Exception(
            "{}\nFor more info on error messages, check: "
            "https://cloud.google.com/apis/design/errors".format(response.error.message)
        )
    
    return response,texts


In [ ]:
response, texts = detect_text("/Users/ineshtandon/Documents/GitHub/NeoBio-CV_Pipeline/testing output images/ocr_prep_debug/test14_stitched.png")

In [ ]:
type(str(response))

In [ ]:
print(response)

In [ ]:
print(texts)

In [ ]:
with open("response.txt", "w") as file:
    file.write(str(response))
    file.write("\n")

## Testing Top Crop Rotation

Top crop rotation to horizontally align individual agent names to mitigate multi block agent name breakdown problem currently present in rotated image.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def rotate_image_bound(image: np.ndarray, angle_deg: float, border_value=(255, 255, 255)) -> np.ndarray:
    """
    Rotate an image without cropping its contents.
    Positive angle = counter-clockwise rotation.
    """
    h, w = image.shape[:2]
    center = (w / 2.0, h / 2.0)

    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)

    cos = abs(M[0, 0])
    sin = abs(M[0, 1])

    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))

    M[0, 2] += (new_w / 2.0) - center[0]
    M[1, 2] += (new_h / 2.0) - center[1]

    rotated = cv2.warpAffine(
        image,
        M,
        (new_w, new_h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border_value,
    )
    return rotated


def pad_to_width(image: np.ndarray, target_width: int, pad_value=255) -> np.ndarray:
    """
    Pad image on left/right to match target width.
    """
    h, w = image.shape[:2]
    if w == target_width:
        return image

    total_pad = target_width - w
    left_pad = total_pad // 2
    right_pad = total_pad - left_pad

    if image.ndim == 2:
        return cv2.copyMakeBorder(
            image, 0, 0, left_pad, right_pad,
            borderType=cv2.BORDER_CONSTANT,
            value=pad_value
        )
    else:
        return cv2.copyMakeBorder(
            image, 0, 0, left_pad, right_pad,
            borderType=cv2.BORDER_CONSTANT,
            value=(pad_value, pad_value, pad_value)
        )


def build_rotated_top_stitched_image(
    image_bgr: np.ndarray,
    roi: tuple,
    *,
    top_extra_bottom_px: int = 6,
    bottom_extra_top_px: int = 6,
    top_rotation_deg: float = 52.0,
    gap_px: int = 20,
    gap_value: int = 255,
):
    """
    Parameters
    ----------
    image_bgr : np.ndarray
        Original BGR image.
    roi : tuple
        (x1, y1, x2, y2) in full-image coordinates.
        Only y1 and y2 are used here.
    top_extra_bottom_px : int
        Extend the top crop slightly downward into the ROI if needed.
    bottom_extra_top_px : int
        Extend the bottom crop slightly upward into the ROI if needed.
    top_rotation_deg : float
        Angle to rotate ONLY the top crop.
        Positive angle = counter-clockwise.
    gap_px : int
        Vertical white separator between top and bottom stitched regions.
    gap_value : int
        Background value for separator/padding.

    Returns
    -------
    dict with:
        top_crop
        top_rotated
        bottom_crop
        stitched
        stitch_boundary_y
        top_bounds
        bottom_bounds
    """
    if image_bgr is None or image_bgr.size == 0:
        raise ValueError("image_bgr is empty or invalid.")

    if len(roi) != 4:
        raise ValueError("roi must be a 4-tuple: (x1, y1, x2, y2)")

    h, w = image_bgr.shape[:2]
    _, roi_y1, _, roi_y2 = roi

    roi_y1 = int(np.clip(roi_y1, 0, h))
    roi_y2 = int(np.clip(roi_y2, 0, h))

    top_y1 = 0
    top_y2 = int(np.clip(roi_y1 + top_extra_bottom_px, 0, h))

    bottom_y1 = int(np.clip(roi_y2 - bottom_extra_top_px, 0, h))
    bottom_y2 = h

    if top_y2 <= top_y1:
        raise ValueError(f"Top crop is empty. Computed bounds: {(0, top_y1, w, top_y2)}")
    if bottom_y2 <= bottom_y1:
        raise ValueError(f"Bottom crop is empty. Computed bounds: {(0, bottom_y1, w, bottom_y2)}")

    top_crop = image_bgr[top_y1:top_y2, 0:w].copy()
    bottom_crop = image_bgr[bottom_y1:bottom_y2, 0:w].copy()

    top_rotated = rotate_image_bound(top_crop, top_rotation_deg, border_value=(gap_value, gap_value, gap_value))

    target_width = max(top_rotated.shape[1], bottom_crop.shape[1])
    top_rotated_padded = pad_to_width(top_rotated, target_width, pad_value=gap_value)
    bottom_crop_padded = pad_to_width(bottom_crop, target_width, pad_value=gap_value)

    if image_bgr.ndim == 2:
        gap = np.full((gap_px, target_width), gap_value, dtype=image_bgr.dtype)
    else:
        gap = np.full((gap_px, target_width, 3), gap_value, dtype=image_bgr.dtype)

    stitched = np.vstack([top_rotated_padded, gap, bottom_crop_padded])

    stitch_boundary_y = top_rotated_padded.shape[0]

    return {
        "top_crop": top_crop,
        "top_rotated": top_rotated_padded,
        "bottom_crop": bottom_crop_padded,
        "stitched": stitched,
        "stitch_boundary_y": stitch_boundary_y,
        "top_bounds": (0, top_y1, w, top_y2),
        "bottom_bounds": (0, bottom_y1, w, bottom_y2),
    }

In [ ]:
from pathlib import Path
import sys
# Resolve project root and add its src/ directory to sys.path
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "neobio").exists()),
    None,
)
if project_root is None:
    raise ModuleNotFoundError("Could not find 'src/neobio' from the current notebook location.")

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from neobio.blot.lane_mask import build_lane_mask, roi_from_mask
# _ = (build_lane_mask, roi_from_mask)

In [ ]:

# -------------------------
# Example usage
# -------------------------

# 1) Load image
image_path = "/Users/ineshtandon/Documents/GitHub/NeoBio-CV_Pipeline/NeoBio Input Images/test14.jpeg"
image_bgr = cv2.imread(image_path)
if image_bgr is None:
    raise FileNotFoundError(f"Could not read image: {image_path}")

# 2) Put your ROI here from the blot pipeline
#    Format: (x1, y1, x2, y2)
#    Replace this example with your real ROI.
roi = roi_from_mask(build_lane_mask(image_bgr))

# 3) Build stitched OCR input
result = build_rotated_top_stitched_image(
    image_bgr,
    roi,
    top_extra_bottom_px=10,
    bottom_extra_top_px=10,
    top_rotation_deg=-52.2,   # try 48, 50, 52, 54, 56 if needed
    gap_px=20,
    gap_value=255,
)

stitched_bgr = result["stitched"]

# 4) Save if you want
# output_path = "/testing output images/ocr_prep_debug/test14_rotated_top_stitched.png"
# cv2.imwrite(output_path, stitched_bgr)
cv2.imwrite('./rotation_test.png',stitched_bgr)

# 5) Show results
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")

axes[1].imshow(cv2.cvtColor(result["top_crop"], cv2.COLOR_BGR2RGB))
axes[1].set_title("Top Crop")

axes[2].imshow(cv2.cvtColor(result["top_rotated"], cv2.COLOR_BGR2RGB))
axes[2].set_title("Rotated Top Crop")

axes[3].imshow(cv2.cvtColor(stitched_bgr, cv2.COLOR_BGR2RGB))
axes[3].axhline(result["stitch_boundary_y"], linestyle="--")
axes[3].set_title("Final Stitched OCR Input")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Top bounds:", result["top_bounds"])
print("Bottom bounds:", result["bottom_bounds"])
print("Stitch boundary y:", result["stitch_boundary_y"])
# print("Saved to:", output_path)

In [ ]:
response, texts = detect_text("/Users/ineshtandon/Documents/GitHub/NeoBio-CV_Pipeline/notebooks/rotation_test.png")

In [ ]:
with open("response.txt", "w") as file:
    file.write(str(response))
    file.write("\n")